<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/Section_7_ARn_models_with_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Model for AR analysis - to be used as reference for signal processing AR

In [112]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.graphics.tsaplots as sgt
import statsmodels.tsa.stattools as sts
from statsmodels.tsa.seasonal import seasonal_decompose
import seaborn as sns
sns.set()

In [113]:
# pip install numpy pandas plotly

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def simulate_ar(n: int, phi: np.ndarray, sigma2: float, seed: int = 123) -> np.ndarray:
    rng = np.random.default_rng(seed)
    p = len(phi)
    x = np.zeros(n, dtype=float)
    eps = rng.normal(0.0, np.sqrt(sigma2), size=n)

    # rozruch
    for k in range(p, n):
        x[k] = np.dot(phi, x[k-p:k][::-1]) + eps[k]
    return x

def generate_ts_smart(
    n=2000,
    noise_power=0.6,          # wariancja białego szumu dodawanego do sumy
    ar_phi=(0.65, -0.25, 0.18, 0.10),  # "prawdziwy" AR(4) do wykrycia przez PACF
    ar_noise_power=0.4,       # wariancja pobudzenia AR
    seasonal_period=48,       # okres sezonowości w próbkach
    seasonal_amp=3.0,         # skala sezonowości
    seed=123
):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    # 1) Trend: lekko nieliniowy + skok poziomu (żeby detrend było widać)
    trend = 0.002 * t + 2e-6 * (t - n/2)**2
    level_shift = np.where(t >= int(0.6*n), 3.0, 0.0)  # skok
    trend = trend + level_shift

    # 2) Sezonowość: powtarzalny "profil" (nie sinus) + modulacja amplitudy
    k = np.arange(seasonal_period)
    profile = (
        1.8 * np.exp(-0.5 * ((k - 0.25*seasonal_period)/(0.10*seasonal_period))**2)  # wąski pik
        - 0.9 * np.exp(-0.5 * ((k - 0.70*seasonal_period)/(0.18*seasonal_period))**2) # szeroka dolina
        + 0.2 * ((k/seasonal_period) - 0.5)  # lekka asymetria (rampa)
    )
    profile = (profile - profile.mean()) / (profile.std() + 1e-12)  # standaryzuj profil

    amp_mod = 1.0 + 0.25 * np.sin(2*np.pi * t / (10*seasonal_period))  # wolna modulacja amplitudy
    season = seasonal_amp * amp_mod * profile[t % seasonal_period]

    # 3) Składnik AR(p) - to potem ma wyjść w PACF i dać dobry AR(n)
    ar_part = simulate_ar(n, np.array(ar_phi, dtype=float), sigma2=ar_noise_power, seed=seed+1)

    # 4) Biały szum (sterujesz noise_power)
    noise = rng.normal(0.0, np.sqrt(noise_power), size=n)

    y = trend + season + ar_part + noise

    df = pd.DataFrame({
        "k": t,
        "y": y,
        "trend": trend,
        "season": season,
        "ar_part": ar_part,
        "noise": noise
    })
    return df

df = generate_ts_smart(
    n=2000,
    noise_power=0.6,
    ar_phi=(0.65, -0.25, 0.18, 0.10),
    ar_noise_power=0.4,
    seasonal_period=48,
    seasonal_amp=3.0,
    seed=123
)

# ---- wykres (dark) ----
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=["y (trend + season(profile) + AR + noise)", "Trend + Season", "AR part (ukryty składnik)"]
)

fig.add_trace(go.Scatter(x=df["k"], y=df["y"], mode="lines", name="y"), row=1, col=1)

fig.add_trace(go.Scatter(x=df["k"], y=df["trend"], mode="lines", name="trend"), row=2, col=1)
fig.add_trace(go.Scatter(x=df["k"], y=df["season"], mode="lines", name="season"), row=2, col=1)

fig.add_trace(go.Scatter(x=df["k"], y=df["ar_part"], mode="lines", name="ar_part"), row=3, col=1)

fig.update_layout(template="plotly_dark", height=900, width=1100, legend=dict(orientation="h"))
fig.update_xaxes(title_text="k", row=3, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.show()


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   k        2000 non-null   int64  
 1   y        2000 non-null   float64
 2   trend    2000 non-null   float64
 3   season   2000 non-null   float64
 4   ar_part  2000 non-null   float64
 5   noise    2000 non-null   float64
dtypes: float64(5), int64(1)
memory usage: 93.9 KB


In [115]:
import pandas as pd

# zakładam, że masz już df z kolumną "y" (np. z generate_ts_smart)

df_y = df.copy()

# Time as Index (minutowe próbkowanie)
df_y["Time"] = pd.date_range(
    start="2026-01-01 00:00:00",  # dowolny start
    periods=len(df_y),
    freq="min"
)

df_y["Time"] = pd.to_datetime(df_y["Time"])   # tu akurat już jest datetime, ale trzymamy styl jak w wzorcu
df_y.set_index("Time", inplace=True)

df_y = df_y[["y"]]            # wytnij niepotrzebne kolumny (zostaje tylko wynik)
df_y = df_y.asfreq("min")     # data frequency as 'minute'
df_y = df_y.ffill()           # fill NaN forward (gdyby asfreq coś wstawił)

df_y.head()


,y
Time,
2026-01-01 00:00:00,0.930093
2026-01-01 00:01:00,1.607782
2026-01-01 00:02:00,3.174741
2026-01-01 00:03:00,2.721027
2026-01-01 00:04:00,4.314876


In [116]:
df_y.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2000 entries, 2026-01-01 00:00:00 to 2026-01-02 09:19:00
Freq: min
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   y       2000 non-null   float64
dtypes: float64(1)
memory usage: 31.2 KB


In [117]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_y.index, y=df_y["y"], mode="lines", name="y"))

fig.update_layout(
    template="plotly_dark",
    title="y(t) — 1-min sampling",
    xaxis_title="Time",
    yaxis_title="y"
)

fig.show()


ACF dla sezonowości

In [118]:
# pip install numpy statsmodels plotly

import numpy as np
from statsmodels.tsa.stattools import acf
import plotly.graph_objects as go

# --- wejście ---
y = df_y["y"].astype(float).values

# --- parametry ACF ---
max_lag = 600      # ile lagów liczysz (minuty)
min_period = 10    # min sensowny okres
max_period = 300   # max sensowny okres

acf_vals = acf(y, nlags=max_lag, fft=True)
lags = np.arange(max_lag + 1)

# --- estymacja okresu z ACF (RAW) ---
lo, hi = min_period, min(max_period, max_lag)
search = acf_vals[lo:hi+1]
period_est = int(lo + np.argmax(search))

top_k = 5
top_idx = np.argsort(search)[::-1][:top_k]
top_periods = (lo + top_idx).tolist()
top_vals = search[top_idx].tolist()

print(f"Estimated period from RAW ACF: {period_est} samples (minutes)")
print("Top candidates:", list(zip(top_periods, [round(v,4) for v in top_vals])))

# --- wykres ACF (Plotly dark) ---
fig = go.Figure()
fig.add_trace(go.Bar(x=lags, y=acf_vals, name="ACF(y)"))
fig.add_hline(y=0)
fig.add_vline(x=period_est)

fig.update_layout(
    template="plotly_dark",
    title=f"ACF(y) — RAW | period_est={period_est}",
    xaxis_title="lag [min]",
    yaxis_title="ACF",
    height=500,
    width=1100,
    showlegend=False
)
fig.show()


Estimated period from RAW ACF: 48 samples (minutes)
Top candidates: [(48, 0.8845), (47, 0.8789), (49, 0.876), (46, 0.8598), (50, 0.856)]


Detrening - usunięcie sezonowości

estymowny period wpisz w zmienną: seasonal_period

In [119]:
# pip install statsmodels plotly

import pandas as pd
from statsmodels.tsa.seasonal import STL
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# df_y: DataFrame z DateTimeIndex (co minutę) i jedną kolumną: "y"

seasonal_period = 48  # <- u

y = df_y["y"].astype(float)

stl = STL(y, period=seasonal_period, robust=True)
stl_res = stl.fit()

# składowe
y_trend   = stl_res.trend.rename("trend")
y_season  = stl_res.seasonal.rename("seasonal")
y_resid   = stl_res.resid.rename("resid")                 # to jest "po detrend i po sezonowości"
y_detr    = (y - y_trend).rename("y_detrended")           # usunięty trend, sezon zostaje
y_deseas  = (y - y_season).rename("y_deseasonalized")     # usunięta sezonowość, trend zostaje

# (opcjonalnie) wrzuć do jednego DF, bo potem przy ACF/PACF wygodniej
df_clean = pd.concat([y, y_trend, y_season, y_detr, y_deseas, y_resid], axis=1)

# ----------------------------
# Wykresy (Plotly dark)
# ----------------------------
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=[
        "Oryginał y oraz trend (STL)",
        "Sezonowość (STL)",
        "y po detrend (y - trend) i po deseason (y - seasonal)",
        "Residual (y - trend - seasonal)  — dla modeli ACF/PACF/AR"
    ]
)

# 1) y + trend
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y"], mode="lines", name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["trend"], mode="lines", name="trend"), row=1, col=1)

# 2) seasonal
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["seasonal"], mode="lines", name="seasonal"), row=2, col=1)

# 3) detrended + deseasonalized
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y_detrended"], mode="lines", name="y_detrended"), row=3, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y_deseasonalized"], mode="lines", name="y_deseasonalized"), row=3, col=1)

# 4) residual
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["resid"], mode="lines", name="resid"), row=4, col=1)

fig.update_layout(template="plotly_dark", height=1050, width=1200, legend=dict(orientation="h"))
fig.update_xaxes(title_text="Time", row=4, col=1)
fig.show()

# df_clean.head()


In [120]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2000 entries, 2026-01-01 00:00:00 to 2026-01-02 09:19:00
Freq: min
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   y                 2000 non-null   float64
 1   trend             2000 non-null   float64
 2   seasonal          2000 non-null   float64
 3   y_detrended       2000 non-null   float64
 4   y_deseasonalized  2000 non-null   float64
 5   resid             2000 non-null   float64
dtypes: float64(6)
memory usage: 109.4 KB


ADF - test

In [121]:
import statsmodels.tsa.stattools as sts  # ADF test lives here

# --- columns to test (choose what makes sense) ---
cols = ["y", "y_detrended", "y_deseasonalized", "resid"]
alpha = 0.05

# sanity: keep only columns that exist
cols = [c for c in cols if c in df_clean.columns]

print(f"Running ADF for {len(cols)} columns: {cols}")

for col in cols:
    print("\n" + "=" * 80)
    print(f"ADF test | column: {col}")

    series = df_clean[col].dropna()

    adf_out = sts.adfuller(series, autolag="AIC")

    adf_stat = adf_out[0]
    p_value  = adf_out[1]
    used_lags = adf_out[2]
    n_obs = adf_out[3]
    crit = adf_out[4]
    icbest = adf_out[5]

    print(f"ADF statistic: {adf_stat:.6f}")
    print(f"p-value:       {p_value:.6g}")
    print(f"used lags:     {used_lags}")
    print(f"n obs:         {n_obs}")
    print(f"critical values: 1%={crit['1%']:.6f}, 5%={crit['5%']:.6f}, 10%={crit['10%']:.6f}")
    print(f"IC best:       {icbest:.6f}")

    if p_value < alpha:
        print(f"Decision @ {alpha}: REJECT H0 (unit root) -> series is likely STATIONARY.")
    else:
        print(f"Decision @ {alpha}: FAIL TO REJECT H0 -> series is likely NON-STATIONARY (unit root).")

    if adf_stat < crit["5%"]:
        print("Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.")
    else:
        print("Sanity check: ADF statistic is NOT below the 5% critical value -> supports non-stationarity.")


Running ADF for 4 columns: ['y', 'y_detrended', 'y_deseasonalized', 'resid']

ADF test | column: y
ADF statistic: -4.986682
p-value:       2.35803e-05
used lags:     26
n obs:         1973
critical values: 1%=-3.433669, 5%=-2.863006, 10%=-2.567550
IC best:       6657.762362
Decision @ 0.05: REJECT H0 (unit root) -> series is likely STATIONARY.
Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.

ADF test | column: y_detrended
ADF statistic: -18.688363
p-value:       2.04e-30
used lags:     26
n obs:         1973
critical values: 1%=-3.433669, 5%=-2.863006, 10%=-2.567550
IC best:       6356.158267
Decision @ 0.05: REJECT H0 (unit root) -> series is likely STATIONARY.
Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.

ADF test | column: y_deseasonalized
ADF statistic: -0.569620
p-value:       0.8777
used lags:     19
n obs:         1980
critical values: 1%=-3.433657, 5%=-2.863001, 10%=-2.567548
IC best:       5330.166427

#ADF
Biorę już tylko resid dla dalszych bo spełnia stacjonarność H0 -> Dickeya Fullera

In [122]:
import statsmodels.tsa.stattools as sts

alpha = 0.05
series = df_clean["resid"].dropna()

adf_stat, p_value, used_lags, n_obs, crit, icbest = sts.adfuller(series, autolag="AIC")

print("ADF test | series: resid")
print(f"ADF statistic: {adf_stat:.6f}")
print(f"p-value:       {p_value:.6g}")
print(f"used lags:     {used_lags}")
print(f"n obs:         {n_obs}")
print(f"critical values: 1%={crit['1%']:.6f}, 5%={crit['5%']:.6f}, 10%={crit['10%']:.6f}")
print(f"IC best:       {icbest:.6f}")

if p_value < alpha:
    print(f"Decision @ {alpha}: REJECT H0 (unit root) -> resid is likely STATIONARY.")
else:
    print(f"Decision @ {alpha}: FAIL TO REJECT H0 -> resid is likely NON-STATIONARY.")


ADF test | series: resid
ADF statistic: -10.747159
p-value:       2.72322e-19
used lags:     22
n obs:         1977
critical values: 1%=-3.433662, 5%=-2.863003, 10%=-2.567549
IC best:       5195.714024
Decision @ 0.05: REJECT H0 (unit root) -> resid is likely STATIONARY.


PACF

In [123]:
# ===== PACF(resid) + 95% CI band + wybór rzędu AR(p) =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from scipy.stats import norm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 0) Wejście: df_y (DateTimeIndex co minutę, kolumna "y")
# ---------------------------------------------------------
# df_y musi istnieć wcześniej. Jeśli nie, to:
# df_y = ...  # DataFrame z kolumną "y" i indexem datetime freq="min"

# ---------------------------------------------------------
# 1) STL -> df_clean z resid
# ---------------------------------------------------------
seasonal_period = 48  # <-- ustaw (np. z ACF). 48 = 48 minut

y = df_y["y"].astype(float)
stl_res = STL(y, period=seasonal_period, robust=True).fit()

df_clean = pd.concat([
    y.rename("y"),
    stl_res.trend.rename("trend"),
    stl_res.seasonal.rename("seasonal"),
    (y - stl_res.trend).rename("y_detrended"),
    (y - stl_res.seasonal).rename("y_deseasonalized"),
    stl_res.resid.rename("resid")
], axis=1)

# ---------------------------------------------------------
# 2) PACF(resid) + 95% CI (±z/sqrt(N)) + wybór p
# ---------------------------------------------------------
x = df_clean["resid"].dropna().astype(float).values
n = len(x)

max_lag = 40
alpha = 0.05  # 95% CI

pacf_vals = pacf(x, nlags=max_lag, method="ywmle")

z = norm.ppf(1 - alpha/2)
conf = z / np.sqrt(n)  # 95% band around 0

lags = np.arange(max_lag + 1)

sig_lags = np.where(np.abs(pacf_vals[1:]) > conf)[0] + 1  # ignore lag 0
p_ar = int(sig_lags.max()) if len(sig_lags) else 0

print(f"PACF significance band: ±{conf:.6f} (N={n})")
print(f"Significant lags: {sig_lags.tolist()}")
print(f"Chosen AR order p = {p_ar}")

# ---------------------------------------------------------
# 3) Plotly dark: resid + PACF z CI
# ---------------------------------------------------------
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
    subplot_titles=[
        "resid (po STL: y - trend - seasonal)",
        f"PACF(resid) | 95% CI band ±{conf:.4f} | chosen p={p_ar}"
    ]
)

# resid
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["resid"],
    mode="lines", name="resid"
), row=1, col=1)

# PACF bars
fig.add_trace(go.Bar(
    x=lags, y=pacf_vals, name="PACF"
), row=2, col=1)

# CI lines (bardziej czytelne niż prostokąt)
fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y= conf, row=2, col=1)
fig.add_hline(y=-conf, row=2, col=1)

# chosen p
if p_ar > 0:
    fig.add_vline(x=p_ar, row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1200,
    legend=dict(orientation="h")
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.update_yaxes(title_text="PACF", row=2, col=1)

fig.show()


PACF significance band: ±0.043826 (N=2000)
Significant lags: [1, 2, 4, 16, 18, 21]
Chosen AR order p = 21


PACF - sensowny

In [124]:
# ===== PACF(resid) + 95% CI band + wybór rzędu AR(p) =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from scipy.stats import norm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 0) Wejście: df_y (DateTimeIndex co minutę, kolumna "y")
# ---------------------------------------------------------
# df_y musi istnieć wcześniej. Jeśli nie, to:
# df_y = ...  # DataFrame z kolumną "y" i indexem datetime freq="min"

# ---------------------------------------------------------
# 1) STL -> df_clean z resid
# ---------------------------------------------------------
seasonal_period = 48  # <-- ustaw (np. z ACF). 48 = 48 minut

y = df_y["y"].astype(float)
stl_res = STL(y, period=seasonal_period, robust=True).fit()

df_clean = pd.concat([
    y.rename("y"),
    stl_res.trend.rename("trend"),
    stl_res.seasonal.rename("seasonal"),
    (y - stl_res.trend).rename("y_detrended"),
    (y - stl_res.seasonal).rename("y_deseasonalized"),
    stl_res.resid.rename("resid")
], axis=1)

# ---------------------------------------------------------
# 2) PACF(resid) + 95% CI (±z/sqrt(N)) + wybór p
# ---------------------------------------------------------
x = df_clean["resid"].dropna().astype(float).values
n = len(x)

max_lag = 10
alpha = 0.05  # 95% CI

pacf_vals = pacf(x, nlags=max_lag, method="ywmle")

z = norm.ppf(1 - alpha/2)
conf = z / np.sqrt(n)  # 95% band around 0

lags = np.arange(max_lag + 1)

sig_lags = np.where(np.abs(pacf_vals[1:]) > conf)[0] + 1  # ignore lag 0
p_ar = int(sig_lags.max()) if len(sig_lags) else 0

print(f"PACF significance band: ±{conf:.6f} (N={n})")
print(f"Significant lags: {sig_lags.tolist()}")
print(f"Chosen AR order p = {p_ar}")

# ---------------------------------------------------------
# 3) Plotly dark: resid + PACF z CI
# ---------------------------------------------------------
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
    subplot_titles=[
        "resid (po STL: y - trend - seasonal)",
        f"PACF(resid) | 95% CI band ±{conf:.4f} | chosen p={p_ar}"
    ]
)

# resid
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["resid"],
    mode="lines", name="resid"
), row=1, col=1)

# PACF bars
fig.add_trace(go.Bar(
    x=lags, y=pacf_vals, name="PACF"
), row=2, col=1)

# CI lines (bardziej czytelne niż prostokąt)
fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y= conf, row=2, col=1)
fig.add_hline(y=-conf, row=2, col=1)

# chosen p
if p_ar > 0:
    fig.add_vline(x=p_ar, row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1200,
    legend=dict(orientation="h")
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.update_yaxes(title_text="PACF", row=2, col=1)

fig.show()


PACF significance band: ±0.043826 (N=2000)
Significant lags: [1, 2, 4]
Chosen AR order p = 4


Interpretacja (krótko, co z tego czytasz):

ar.L1 (coef, P>|z|): czy masz istotną autokorelację rzędu 1 (p-value małe → AR(1) ma sens).

const (P>|z|): czy średnia serii/reszt ≠ 0 (duże p-value → stała zbędna, resid ~ 0-mean).

AIC/BIC: do porównań modeli (mniejsze = lepiej, ale porównuj tylko modele na tych samych danych).

Ljung-Box Prob(Q): czy reszty po modelu są „białe” (duże p-value → brak autokorelacji → dobrze).

Jarque–Bera Prob(JB): czy reszty są normalne (małe p-value → nienormalne ogony → uwaga na wnioski o rozkładzie/przedziałach).

In [125]:
from statsmodels.tsa.arima.model import ARIMA  # import klasy ARIMA ze statsmodels (modelowanie szeregów czasowych)
model_ret_ar_1 = ARIMA(df_clean.resid, order=(1,0,0))  # zdefiniuj model ARIMA(1,0,0)=AR(1) na sygnale resid (p=1, d=0, q=0)


In [126]:
results_ret_ar_1 = model_ret_ar_1.fit()  # dopasuj (wyestymuj) parametry modelu do danych i zwróć obiekt wyników (współczynniki, AIC/BIC, diagnostyka, predykcje)


In [127]:
# wyświetla raport z dopasowania: parametry AR(1), ich istotność, sigma^2 oraz metryki jakości (LL, AIC/BIC) i testy diagnostyczne reszt
print ("Statistic " ,results_ret_ar_1.summary())

Statistic                                 SARIMAX Results                                
Dep. Variable:                  resid   No. Observations:                 2000
Model:                 ARIMA(1, 0, 0)   Log Likelihood               -2631.818
Date:                Tue, 06 Jan 2026   AIC                           5269.637
Time:                        11:31:37   BIC                           5286.440
Sample:                    01-01-2026   HQIC                          5275.806
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0021      0.028     -0.076      0.939      -0.056       0.052
ar.L1          0.2679      0.022     12.197      0.000       0.225       0.311
sigma2         0.8137      0.018     44.5

#interpretacja

ar.L1 p=0.000 → współczynnik AR(1) jest istotnie różny od zera. To jest dowód, że w resid była przewidywalna struktura (zależność od poprzedniej próbki). Czyli istnieje predictive power

const p=0.939 → stała jest nieistotna. To tylko mówi: średnia resid ≈ 0, więc nie musisz mieć wyrazu wolnego.

Ljung-Box Prob(Q)=0.58 → po dopasowaniu modelu reszty wyglądają na nieautokorelowane. To jest dobry znak: model “wyciągnął” przewidywalną część.

Jarque–Bera Prob(JB)=0.00 + kurtosis=5.03 → reszty nie są normalne (cięższe ogony). To wpływa na założenia o rozkładzie błędu / przedziałach ufności, ale nie unieważnia samej predykcji średniej.

#jeszcze raz Dickey - Fuller

In [128]:
sts.adfuller(df_clean.resid)

(np.float64(-10.747158673613605),
 np.float64(2.723222379236657e-19),
 22,
 1977,
 {'1%': np.float64(-3.433661993406868),
  '5%': np.float64(-2.8630030510232647),
  '10%': np.float64(-2.567548867394869)},
 np.float64(5195.714024295296))

## Interpretacja
p-value jest maleńkie -> szereg jest super stacjonarny

Model II-rzędu (patrzymy, czy rozwinięcie będzie miało sens w predykcji)

Ma, stała (const) jest nieistotna - można ją ominąć, współczynniki L1, L2 mają p-value < 0.05 -> można rozwijać

Log Likehood (im większy tym lepszy) -> jeśli rośnie wraz ze wzrostem modelu, to lepsze dopasowanie

In [129]:

model_ret_ar_2 = ARIMA(df_clean.resid, order = (2,0,0))
results_ret_ar_2 = model_ret_ar_2.fit()
results_ret_ar_2.summary()
print ("Statistic " ,results_ret_ar_2.summary())

Statistic                                 SARIMAX Results                                
Dep. Variable:                  resid   No. Observations:                 2000
Model:                 ARIMA(2, 0, 0)   Log Likelihood               -2629.663
Date:                Tue, 06 Jan 2026   AIC                           5267.326
Time:                        11:31:37   BIC                           5289.729
Sample:                    01-01-2026   HQIC                          5275.552
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0021      0.029     -0.074      0.941      -0.059       0.055
ar.L1          0.2554      0.023     11.090      0.000       0.210       0.301
ar.L2          0.0464      0.022      2.1

Sprawdzenie Modeli rzędu 1 i 2 -> AR(1) - AR(2)
Test LLR

In [130]:
from scipy.stats.distributions import chi2  # rozkład chi-kwadrat: potrzebny do p-value w teście ilorazu wiarygodności (LLR)

def LLR_test(mod_1, mod_2, DF=1):          # funkcja porównująca 2 zagnieżdżone modele (mod_2 “większy”), DF = różnica liczby parametrów
    L1 = mod_1.fit().llf                    # dopasuj model 1 i pobierz log-likelihood (LL) z optimum
    L2 = mod_2.fit().llf                    # dopasuj model 2 i pobierz log-likelihood (LL) z optimum
    LR = (2 * (L2 - L1))                    # statystyka LLR: 2*(LL2-LL1); im większa, tym bardziej model 2 wygrywa
    p = chi2.sf(LR, DF).round(3)            # p-value = P(Chi2_DF >= LR); małe p => model 2 istotnie lepszy od modelu 1
    return p                                # zwróć p-value testu LLR


Interpretacja:

Ponieważ 0.038 < 0.05, to:

odrzucasz H0 na poziomie 5%,

czyli AR(2) daje statystycznie istotną poprawę względem AR(1).

Uwaga

To mówi o dopasowaniu (likelihood), nie gwarantuje dużej poprawy predykcji out-of-sample (czasem zysk jest mały).

Przy dużym N łatwo o “istotne” p-value dla małych różnic — dlatego i tak warto spojrzeć na BIC albo na MSE na teście.I

In [131]:
LLR_test(model_ret_ar_1, model_ret_ar_2)

np.float64(0.038)

AR(3)

In [132]:
model_ret_ar_3 = ARIMA(df_clean.resid, order = (3,0,0))
results_ret_ar_3 = model_ret_ar_3.fit()
results_ret_ar_3.summary()
print ("Statistic " ,results_ret_ar_3.summary())

Statistic                                 SARIMAX Results                                
Dep. Variable:                  resid   No. Observations:                 2000
Model:                 ARIMA(3, 0, 0)   Log Likelihood               -2628.065
Date:                Tue, 06 Jan 2026   AIC                           5266.130
Time:                        11:31:38   BIC                           5294.135
Sample:                    01-01-2026   HQIC                          5276.413
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0021      0.030     -0.071      0.943      -0.061       0.057
ar.L1          0.2536      0.023     11.014      0.000       0.208       0.299
ar.L2          0.0362      0.022      1.6

In [133]:
LLR_test(model_ret_ar_2, model_ret_ar_3)

np.float64(0.074)

AR(4) -> tu już p-value wychodzi duże, nie ma sensu robić AR(4)

chociaż LLR wychodzi 0.0 to porównanie AR(3) do AR(4) moze być małe, ale nie mówi o niczym

In [134]:
model_ret_ar_4 = ARIMA(df_clean.resid, order = (4,0,0))
results_ret_ar_4 = model_ret_ar_4.fit()
results_ret_ar_4.summary()
print ("Statistic " ,results_ret_ar_4.summary())

Statistic                                 SARIMAX Results                                
Dep. Variable:                  resid   No. Observations:                 2000
Model:                 ARIMA(4, 0, 0)   Log Likelihood               -2618.768
Date:                Tue, 06 Jan 2026   AIC                           5249.535
Time:                        11:31:40   BIC                           5283.141
Sample:                    01-01-2026   HQIC                          5261.874
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0022      0.033     -0.067      0.947      -0.067       0.063
ar.L1          0.2497      0.023     10.886      0.000       0.205       0.295
ar.L2          0.0327      0.022      1.4

In [135]:
LLR_test(model_ret_ar_3, model_ret_ar_4)

np.float64(0.0)

Prognoza bez "rolling update" -> nie patrzy na ostatni punkt, robi predykcję z modelu tylko

In [136]:
# ===== Diagnostyka reszt prognozy: ACF(error) dla naive vs AR(best_BIC) vs AR(best_MSE) =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
from scipy.stats import norm

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------------
# Wejście: df_clean["resid"] (Twoje resid po STL)
# ------------------------------------------------------------------
y = df_clean["resid"].dropna().astype(float)
n = len(y)

# ------------------------------------------------------------------
# Ustawienia porównania modeli
# ------------------------------------------------------------------
p_list = [1, 2, 3, 4, 5, 6, 8, 10]   # przetestowane rzędy AR(p)
train_frac = 0.8
split = int(train_frac * n)

train = y.iloc[:split]
test  = y.iloc[split:]   # to prognozujemy 1-step ahead

# ------------------------------------------------------------------
# NAIVE baseline (wyjaśnienie poniżej)
# y_hat[t] = y[t-1]  -> "jutro takie jak dziś", "następna próbka jak poprzednia"
# ------------------------------------------------------------------
naive_pred = test.shift(1)                   # prognoza = poprzednia wartość
naive_err = (test - naive_pred).dropna()     # błąd prognozy e_t = y_t - y_hat_t

mse_naive = float(np.mean(naive_err.values**2))

# ------------------------------------------------------------------
# AR(p) na train -> 1-step predykcja na test -> błąd + metryki
# ------------------------------------------------------------------
rows = []
pred_store = {}
err_store = {}

for p in p_list:
    res = ARIMA(train, order=(p, 0, 0), trend="n").fit()

    pred = res.get_prediction(start=test.index[0], end=test.index[-1]).predicted_mean
    err = (test - pred)

    mse = float(np.mean(err.values**2))

    rows.append({"p": p, "AIC": float(res.aic), "BIC": float(res.bic), "MSE_test": mse})
    pred_store[p] = pred
    err_store[p] = err

df_cmp = pd.DataFrame(rows)

best_p_bic = int(df_cmp.loc[df_cmp["BIC"].idxmin(), "p"])
best_p_mse = int(df_cmp.loc[df_cmp["MSE_test"].idxmin(), "p"])

mse_bic = float(df_cmp.loc[df_cmp["p"] == best_p_bic, "MSE_test"].iloc[0])
mse_mse = float(df_cmp.loc[df_cmp["p"] == best_p_mse, "MSE_test"].iloc[0])

print(df_cmp.sort_values("BIC").reset_index(drop=True))
print(f"\nnaive MSE: {mse_naive:.6f}")
print(f"best_BIC: AR({best_p_bic}) | MSE_test={mse_bic:.6f}")
print(f"best_MSE: AR({best_p_mse}) | MSE_test={mse_mse:.6f}")

# ------------------------------------------------------------------
# ACF błędu prognozy: chcemy, żeby error był możliwie "biały"
# (czyli ACF poza lag=0 blisko zera i w pasie ufności)
# ------------------------------------------------------------------
max_lag = 60
alpha = 0.05
z = norm.ppf(1 - alpha/2)

def acf_with_band(x, max_lag, z):
    x = pd.Series(x).dropna().values
    n = len(x)
    acf_vals = acf(x, nlags=max_lag, fft=True)
    conf = z / np.sqrt(n)   # ±1.96/sqrt(N)
    return acf_vals, conf, n

acf_naive, conf_naive, n_naive = acf_with_band(naive_err, max_lag, z)
acf_bic, conf_bic, n_bic = acf_with_band(err_store[best_p_bic], max_lag, z)
acf_mse, conf_mse, n_mse = acf_with_band(err_store[best_p_mse], max_lag, z)

lags = np.arange(max_lag + 1)

# ------------------------------------------------------------------
# Plotly dark: 2 wykresy
# 1) test + predykcje
# 2) ACF(error) dla naive, best_BIC, best_MSE
# ------------------------------------------------------------------
fig = make_subplots(
    rows=2, cols=1, vertical_spacing=0.12,
    subplot_titles=[
        "1-step predictions on resid: test vs naive vs best_BIC vs best_MSE",
        "ACF of prediction error e(t) = y(t) - y_hat(t)  (im bliżej 0 tym lepiej)"
    ]
)

# --- (1) predykcje ---
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines", name="test"), row=1, col=1)

fig.add_trace(go.Scatter(
    x=naive_pred.index[1:], y=naive_pred.iloc[1:].values,
    mode="lines", name=f"naive (MSE={mse_naive:.4f})"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=pred_store[best_p_bic].index, y=pred_store[best_p_bic].values,
    mode="lines", name=f"AR({best_p_bic}) best_BIC (MSE={mse_bic:.4f})"
), row=1, col=1)

if best_p_mse != best_p_bic:
    fig.add_trace(go.Scatter(
        x=pred_store[best_p_mse].index, y=pred_store[best_p_mse].values,
        mode="lines", name=f"AR({best_p_mse}) best_MSE (MSE={mse_mse:.4f})"
    ), row=1, col=1)

fig.add_hline(y=0, row=1, col=1)

# --- (2) ACF błędu ---
# naive
fig.add_trace(go.Scatter(x=lags, y=acf_naive, mode="lines+markers", name=f"ACF err naive (N={n_naive})"), row=2, col=1)
fig.add_hline(y= conf_naive, row=2, col=1)
fig.add_hline(y=-conf_naive, row=2, col=1)

# best_BIC
fig.add_trace(go.Scatter(x=lags, y=acf_bic, mode="lines+markers", name=f"ACF err AR({best_p_bic}) (N={n_bic})"), row=2, col=1)

# best_MSE (jeśli inne)
if best_p_mse != best_p_bic:
    fig.add_trace(go.Scatter(x=lags, y=acf_mse, mode="lines+markers", name=f"ACF err AR({best_p_mse}) (N={n_mse})"), row=2, col=1)

fig.add_hline(y=0, row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=950,
    width=1200,
    legend=dict(orientation="h")
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="resid", row=1, col=1)

fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="ACF(error)", row=2, col=1)

fig.show()


    p          AIC          BIC  MSE_test
0   1  4226.634272  4237.389790  0.836166
1   4  4210.775235  4237.664029  0.834587
2   2  4224.137163  4240.270440  0.835211
3   5  4211.261085  4243.527638  0.834695
4   3  4225.068644  4246.579679  0.834922
5   6  4211.577974  4249.222286  0.834870
6   8  4212.268966  4260.668796  0.834510
7  10  4214.835893  4273.991241  0.833575

naive MSE: 1.274417
best_BIC: AR(1) | MSE_test=0.836166
best_MSE: AR(10) | MSE_test=0.833575


A ta predykcja z rolling out

In [137]:
# ===== Poprawne 1-step ahead (rolling) dla naive i AR(p) + ACF błędu =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
from scipy.stats import norm

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------
# Dane: resid
# ---------------------------
y = df_clean["resid"].dropna().astype(float)
n = len(y)

p_list = [1, 2, 3, 4, 5, 6, 8, 10]
train_frac = 0.8
split = int(train_frac * n)

train = y.iloc[:split]
test  = y.iloc[split:]

# ---------------------------
# NAIVE (baseline) - poprawnie wyrównany
# y_hat[t] = y[t-1]
# dla pierwszego punktu testu używamy ostatniego punktu z train
# ---------------------------
naive_pred = pd.Series(
    data=np.r_[train.iloc[-1], test.iloc[:-1].values],
    index=test.index,
    name="naive_pred"
)
naive_err = (test - naive_pred)
mse_naive = float(np.mean(naive_err.values**2))

# ---------------------------
# Rolling 1-step ahead ARIMA(p,0,0)
# prognoza dla t oparta o dane do t-1 (z prawdziwym update testu)
# ---------------------------
def rolling_1step_ar(train_series: pd.Series, test_series: pd.Series, p: int) -> pd.Series:
    res = ARIMA(train_series, order=(p,0,0), trend="n").fit()
    preds = []
    for t in test_series.values:
        # prognoza na kolejny punkt (1-step)
        preds.append(float(res.forecast(1).iloc[0]))
        # aktualizacja stanu modelem prawdziwą obserwacją z testu
        res = res.append([t], refit=False)
    return pd.Series(preds, index=test_series.index, name=f"AR({p})_pred")

rows = []
pred_store = {}
err_store  = {}

for p in p_list:
    pred = rolling_1step_ar(train, test, p)
    err  = (test - pred)
    mse  = float(np.mean(err.values**2))

    # BIC/AIC liczymy na train (bo rolling to już “procedura predykcji”)
    fit_train = ARIMA(train, order=(p,0,0), trend="n").fit()

    rows.append({"p": p, "AIC": float(fit_train.aic), "BIC": float(fit_train.bic), "MSE_test": mse})
    pred_store[p] = pred
    err_store[p]  = err

df_cmp = pd.DataFrame(rows)

best_p_bic = int(df_cmp.loc[df_cmp["BIC"].idxmin(), "p"])
best_p_mse = int(df_cmp.loc[df_cmp["MSE_test"].idxmin(), "p"])

mse_bic = float(df_cmp.loc[df_cmp["p"] == best_p_bic, "MSE_test"].iloc[0])
mse_mse = float(df_cmp.loc[df_cmp["p"] == best_p_mse, "MSE_test"].iloc[0])

print(df_cmp.sort_values("BIC").reset_index(drop=True))
print(f"\nnaive MSE: {mse_naive:.6f}")
print(f"best_BIC: AR({best_p_bic}) | MSE_test={mse_bic:.6f}")
print(f"best_MSE: AR({best_p_mse}) | MSE_test={mse_mse:.6f}")

# ---------------------------
# ACF błędu prognozy (czy error jest “biały”)
# ---------------------------
max_lag = 60
alpha = 0.05
z = norm.ppf(1 - alpha/2)

def acf_with_band(err: pd.Series, max_lag: int):
    e = err.dropna().values
    N = len(e)
    a = acf(e, nlags=max_lag, fft=True)
    conf = z / np.sqrt(N)
    return a, conf, N

lags = np.arange(max_lag + 1)

acf_naive, conf_naive, N_naive = acf_with_band(naive_err, max_lag)
acf_bic, conf_bic, N_bic       = acf_with_band(err_store[best_p_bic], max_lag)
acf_mse, conf_mse, N_mse       = acf_with_band(err_store[best_p_mse], max_lag)

# ---------------------------
# Plotly dark
# ---------------------------
fig = make_subplots(
    rows=2, cols=1, vertical_spacing=0.12,
    subplot_titles=[
        "1-step ahead predictions on resid (rolling update): test vs naive vs best_BIC vs best_MSE",
        "ACF of prediction error e(t)=y(t)-y_hat(t) (poza pasem ufności = zostało coś do modelowania)"
    ]
)

# (1) predykcje
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines", name="test"), row=1, col=1)
fig.add_trace(go.Scatter(x=naive_pred.index, y=naive_pred.values, mode="lines", name=f"naive (MSE={mse_naive:.4f})"), row=1, col=1)
fig.add_trace(go.Scatter(x=pred_store[best_p_bic].index, y=pred_store[best_p_bic].values, mode="lines",
                         name=f"AR({best_p_bic}) best_BIC (MSE={mse_bic:.4f})"), row=1, col=1)

if best_p_mse != best_p_bic:
    fig.add_trace(go.Scatter(x=pred_store[best_p_mse].index, y=pred_store[best_p_mse].values, mode="lines",
                             name=f"AR({best_p_mse}) best_MSE (MSE={mse_mse:.4f})"), row=1, col=1)

fig.add_hline(y=0, row=1, col=1)

# (2) ACF error + pas ufności (biorę pas z naive dla referencji; N są podobne)
fig.add_trace(go.Scatter(x=lags, y=acf_naive, mode="lines+markers", name=f"ACF err naive (N={N_naive})"), row=2, col=1)
fig.add_trace(go.Scatter(x=lags, y=acf_bic, mode="lines+markers", name=f"ACF err AR({best_p_bic}) (N={N_bic})"), row=2, col=1)
if best_p_mse != best_p_bic:
    fig.add_trace(go.Scatter(x=lags, y=acf_mse, mode="lines+markers", name=f"ACF err AR({best_p_mse}) (N={N_mse})"), row=2, col=1)

fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y= conf_naive, row=2, col=1)
fig.add_hline(y=-conf_naive, row=2, col=1)

fig.update_layout(template="plotly_dark", height=950, width=1200, legend=dict(orientation="h"))
fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="resid", row=1, col=1)
fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="ACF(error)", row=2, col=1)
fig.show()


    p          AIC          BIC  MSE_test
0   1  4226.634272  4237.389790  0.790003
1   4  4210.775235  4237.664029  0.782820
2   2  4224.137163  4240.270440  0.790612
3   5  4211.261085  4243.527638  0.782292
4   3  4225.068644  4246.579679  0.787106
5   6  4211.577974  4249.222286  0.785354
6   8  4212.268966  4260.668796  0.781390
7  10  4214.835893  4273.991241  0.779611

naive MSE: 1.271479
best_BIC: AR(1) | MSE_test=0.790003
best_MSE: AR(10) | MSE_test=0.779611


# Normalizing

In [138]:
benchmark = df_clean.resid.iloc[0]  # pierwsza wartość sygnału resid jako wartość referencyjna (baseline/benchmark) do porównań w dalszych obliczeniach
#można brać jakąkolwiek wartość szeregu do benchmarku

In [139]:
df_clean['norm'] = df_clean.resid.div(benchmark).mul(100)  # znormalizuj resid względem benchmarku: (resid / benchmark) * 100 -> wynik w [%] benchmarku


In [140]:
df_clean.head()

,y,trend,seasonal,y_detrended,y_deseasonalized,resid,norm
Time,,,,,,,
2026-01-01 00:00:00,0.930093,1.813249,-0.800736,-0.883156,1.730828,-0.082421,100.000000
2026-01-01 00:01:00,1.607782,1.812486,-0.149645,-0.204705,1.757427,-0.055060,66.803329
2026-01-01 00:02:00,3.174741,1.811656,1.361355,1.363085,1.813386,0.001729,-2.098279
2026-01-01 00:03:00,2.721027,1.810756,1.381015,0.910270,1.340012,-0.470745,571.148245
2026-01-01 00:04:00,4.314876,1.809785,2.125859,2.505092,2.189018,0.379233,-460.118319


In [141]:
import plotly.graph_objects as go

# 1) y
fig_y = go.Figure()
fig_y.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y"], mode="lines", name="y"))
fig_y.update_layout(template="plotly_dark", title="y(t)", xaxis_title="Time", yaxis_title="y", height=450, width=1100)
fig_y.show()

# 2) resid
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(x=df_clean.index, y=df_clean["resid"], mode="lines", name="resid"))
fig_r.add_hline(y=0)
fig_r.update_layout(template="plotly_dark", title="resid(t)", xaxis_title="Time", yaxis_title="resid", height=450, width=1100)
fig_r.show()

# 3) norm
fig_n = go.Figure()
fig_n.add_trace(go.Scatter(x=df_clean.index, y=df_clean["norm"], mode="lines", name="norm [%]"))
fig_n.add_hline(y=100)  # 100% = benchmark
fig_n.update_layout(template="plotly_dark", title="norm(t) = resid/benchmark * 100", xaxis_title="Time", yaxis_title="norm [%]", height=450, width=1100)
fig_n.show()


Interpretacja - norm

100% → dokładnie tyle co benchmark (tu: pierwsza próbka resid).

50% → połowa benchmarku.

200% → dwa razy większe od benchmarku.

−100% → taka sama wartość bezwzględna, ale przeciwny znak.

Sprawdzamy czy wartości norm są cos warte (czy są stacjonarne)

In [142]:
sts.adfuller(df_clean.norm)

(np.float64(-10.7471586736136),
 np.float64(2.7232223792367546e-19),
 22,
 1977,
 {'1%': np.float64(-3.433661993406868),
  '5%': np.float64(-2.8630030510232647),
  '10%': np.float64(-2.567548867394869)},
 np.float64(33216.608348694936))

In [143]:
model_norm_ar_1 = ARIMA(df_clean.norm, order = (1,0,0))

results_norm_ar_1 = model_norm_ar_1.fit()
print(results_norm_ar_1.summary())

                               SARIMAX Results                                
Dep. Variable:                   norm   No. Observations:                 2000
Model:                 ARIMA(1, 0, 0)   Log Likelihood              -16833.995
Date:                Tue, 06 Jan 2026   AIC                          33673.990
Time:                        11:33:51   BIC                          33690.793
Sample:                    01-01-2026   HQIC                         33680.160
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.4257     33.494      0.072      0.942     -63.222      68.073
ar.L1          0.2679      0.022     12.193      0.000       0.225       0.311
sigma2      1.198e+06   2.69e+04     44.567      0.0

In [144]:
model_norm_ar_2 = ARIMA(df_clean.norm, order = (2,0,0))

results_norm_ar_2 = model_norm_ar_2.fit()
print(results_norm_ar_2.summary())

                               SARIMAX Results                                
Dep. Variable:                   norm   No. Observations:                 2000
Model:                 ARIMA(2, 0, 0)   Log Likelihood              -16831.840
Date:                Tue, 06 Jan 2026   AIC                          33671.680
Time:                        11:33:52   BIC                          33694.083
Sample:                    01-01-2026   HQIC                         33679.906
                         - 01-02-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.4257     35.152      0.069      0.945     -66.470      71.322
ar.L1          0.2555      0.023     11.081      0.000       0.210       0.301
ar.L2          0.0464      0.022      2.135      0.0

In [145]:
LLR_test(model_norm_ar_1, model_norm_ar_2)

np.float64(0.038)

## Analysing the Residuals

czyli błąd modelu - dla ostatniego poprawnego rzędu AR - czyli w naszym przypadku AR(3)
AR(4) chociaż z PACF wyszedł 4, to statystycznie lepszy okazała się AR(3)

In [146]:
df_clean['res_ret'] = results_ret_ar_3.resid  # dodaj do df_clean kolumnę z resztami (błędami dopasowania) z modelu ARIMA/AR(3): e_t = y_t - ŷ_t


In [147]:
df_clean.res_ret.mean()                      # policz średnią reszt; wynik ~0 oznacza brak stałego biasu (model nie jest systematycznie “za wysoki” ani “za niski”)
#im bliżej '0' tym lepszy


np.float64(1.6797632775174874e-05)

In [148]:
df_clean.res_ret.var() # im mniejsza tym lepsza

0.8111085930049065

In [149]:
sts.adfuller(df_clean.res_ret) #p-value super

(np.float64(-16.072440393481195),
 np.float64(5.488354888522589e-29),
 5,
 1994,
 {'1%': np.float64(-3.4336337202771823),
  '5%': np.float64(-2.8629905684254977),
  '10%': np.float64(-2.5675422210261676)},
 np.float64(5195.617643791793))

#UWAGA - adfuller dla res_ret powyżej

jeśli p-value dla residuuów z reszt ostatniego modelu AR(n)  wyjdzie > 0,05 to model nie został dobrze przygotwany, istnieje sezonowość, bądź trend

In [150]:
# pip install numpy statsmodels plotly scipy

import numpy as np
from scipy.stats import norm
from statsmodels.tsa.stattools import acf
import plotly.graph_objects as go

series = df_clean["res_ret"].dropna().astype(float).values
n = len(series)

lags_n = 40
acf_vals = acf(series, nlags=lags_n, fft=True)

# zero=False -> pomijamy lag=0
lags = np.arange(1, lags_n + 1)
acf_no0 = acf_vals[1:]

# 95% CI ~ ± z/sqrt(N)
alpha = 0.05
z = norm.ppf(1 - alpha/2)
conf = z / np.sqrt(n)

fig = go.Figure()
fig.add_trace(go.Bar(x=lags, y=acf_no0, name="ACF (no lag 0)"))

fig.add_hline(y=0)
fig.add_hline(y= conf)
fig.add_hline(y=-conf)

fig.update_layout(
    template="plotly_dark",
    title=f"ACF Of Residuals (res_ret) | lags={lags_n} | 95% CI ±{conf:.4f}",
    xaxis_title="lag",
    yaxis_title="ACF",
    height=520,
    width=1100,
    showlegend=False
)

fig.show()


4 lag i 16 lag są poza przedziałem ufności, lecz wartość ACF dla residuuw jest b. mała

In [151]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_clean.index,
    y=df_clean["res_ret"],
    mode="lines",
    name="res_ret"
))

fig.add_hline(y=0)

fig.update_layout(
    template="plotly_dark",
    title="Residuals of Returns (res_ret)",
    xaxis_title="Time",
    yaxis_title="res_ret",
    height=450,
    width=1100,
    showlegend=False
)

fig.show()
